# 🧠♟️ P.O.W.N. — Pawn Optimized With NNUE

Un nouveau réseau d'évaluation, dans l'esprit du NNUE de Stockfish : **HalfKAv2_hm**
(encodage roi + case + pièce, avec l'astuce du miroir gauche-droite), pensé pour être
**beaucoup plus rapide** que PAWN/SPAWN/PWN une fois porté en C++ (mise à jour incrémentale
à chaque coup, au lieu de tout recalculer).

| | |
|---|---|
| Encodage | HalfKAv2_hm — 22 528 combinaisons possibles (roi × case × pièce) |
| Réseau | 22 528 → 256 (×2 perspectives) → 32 → 32 → 1 |
| Paramètres | **≈ 5,78 millions** |
| Données | dump officiel Lichess (FEN + évaluations Stockfish), en streaming |
| Objectif | ≈ 100 millions de positions, 2-3 passages |

⚠️ Cette v1 est volontairement simple (encodage "maison", pas un port exact du NNUE de
Stockfish) : l'important est que l'entraînement et l'inférence utilisent la même formule,
pas de coller au bit près à l'implémentation officielle.

## ⚙️ Spécifique à Colab (différent de Kaggle)

- **Monte ton Drive** (cellule ci-dessous) : contrairement à Kaggle, Colab ne garde rien
  automatiquement. Sans ça, tout est perdu si la session se déconnecte.
- **Limite de temps** (`TIME_MIN`) : l'entraînement s'arrête proprement à l'heure fixée et
  sauvegarde un point de contrôle. Pour continuer un autre jour, remets le chemin de ce
  point de contrôle dans `DOSSIER_DEPART`.
- Le clé Hugging Face (`HF_TOKEN`, dans les **Secrets** de Colab, icône 🔑) n'est **pas
  nécessaire** ici : on télécharge directement le fichier public de Lichess, sans compte.
  Le code la lit quand même si elle existe, pour un futur notebook qui en aurait besoin.

In [ ]:
!pip install -q chess zstandard

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
SORTIE = "/content/drive/MyDrive/pown"
os.makedirs(SORTIE, exist_ok=True)
print("les points de contrôle seront sauvegardés dans :", SORTIE)

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN chargé (pas indispensable pour ce notebook)")
except Exception:
    print("pas de HF_TOKEN configuré — sans importance, on ne télécharge rien depuis Hugging Face ici")

In [ ]:
%%writefile commun_pown.py
"""Code commun de P.O.W.N. (Pawn Optimized With NNUE).

Encodage HalfKAv2_hm (comme le NNUE de Stockfish, avec l'astuce du miroir gauche-droite),
réseau, flux de données (dump officiel Lichess) et boucle d'entraînement.
"""
import io
import json
import math
import os
import time

import chess
import torch
from torch import nn
from torch.utils.data import IterableDataset

# -- encodage HalfKAv2_hm -------------------------------------------------------------
# Pour une perspective donnée : le roi (a-d après miroir) choisit un "bucket" (32 possibles),
# puis chaque autre pièce (sauf notre roi, implicite dans le bucket) ajoute une combinaison
# (case, type de pièce). 11 types : 5 des nôtres (sans roi) + 6 de l'adversaire (avec son roi).
NUM_BUCKETS = 32
NUM_TYPES = 11
F = NUM_BUCKETS * 64 * NUM_TYPES  # 22 528


def _bucket(roi, miroir):
    f = chess.square_file(roi)
    if miroir:
        f = 7 - f
    return chess.square_rank(roi) * 4 + f


def features(board, couleur):
    """Liste des index actifs pour le camp `couleur`, vus depuis son propre roi."""
    roi = board.king(couleur)
    miroir = chess.square_file(roi) >= 4
    b = _bucket(roi, miroir)
    idx = []
    for case, piece in board.piece_map().items():
        if piece.piece_type == chess.KING and piece.color == couleur:
            continue  # notre roi : déjà dans le bucket, pas la peine de le redire
        f, r = chess.square_file(case), chess.square_rank(case)
        if miroir:
            f = 7 - f
        case_vue = r * 8 + f
        nous = piece.color == couleur
        t = (piece.piece_type - 1) if nous else 5 + (piece.piece_type - 1)
        idx.append(b * (64 * NUM_TYPES) + case_vue * NUM_TYPES + t)
    return idx


def exemple(fen, cp):
    """(idx camp au trait, idx camp adverse, cible sigmoid(cp/400)) — cp du point de vue du trait."""
    if len(fen.split(" ")) < 6:
        fen += " 0 1"  # le dump Lichess tronque le FEN aux 4 premiers champs
    board = chess.Board(fen)
    stm = board.turn
    cible = 1.0 / (1.0 + math.exp(-max(-3000, min(3000, cp)) / 400))
    return features(board, stm), features(board, not stm), cible


# -- réseau ----------------------------------------------------------------------------

class ReseauPOWN(nn.Module):
    def __init__(self, f=F, h=256):
        super().__init__()
        self.transformateur = nn.EmbeddingBag(f, h, mode="sum")
        self.c1 = nn.Linear(2 * h, 32)
        self.c2 = nn.Linear(32, 32)
        self.sortie = nn.Linear(32, 1)

    def forward(self, idx_nous, decal_nous, idx_eux, decal_eux):
        a = self.transformateur(idx_nous, decal_nous)
        b = self.transformateur(idx_eux, decal_eux)
        x = torch.clamp(torch.cat([a, b], dim=1), 0, 1)  # "clipped relu" façon NNUE
        x = torch.clamp(self.c1(x), 0, 1)
        x = torch.clamp(self.c2(x), 0, 1)
        return self.sortie(x).squeeze(-1)

    def eval_cp(self, fen):
        """Évalue une position (cp, point de vue du joueur au trait) — pour les tests à la main."""
        board = chess.Board(fen)
        n, e = features(board, board.turn), features(board, not board.turn)
        with torch.no_grad():
            logit = self.forward(torch.tensor(n), torch.tensor([0]), torch.tensor(e), torch.tensor([0]))
        return 400 * math.log(1 / (1 - torch.sigmoid(logit).item() + 1e-9) - 1) * -1


def rassembler(lot):
    """collate_fn : construit les tenseurs plats + décalages attendus par EmbeddingBag."""
    idx_nous, decal_nous, idx_eux, decal_eux, cibles = [], [0], [], [0], []
    for n, e, c in lot:
        idx_nous += n
        decal_nous.append(decal_nous[-1] + len(n))
        idx_eux += e
        decal_eux.append(decal_eux[-1] + len(e))
        cibles.append(c)
    return (torch.tensor(idx_nous, dtype=torch.long), torch.tensor(decal_nous[:-1], dtype=torch.long),
            torch.tensor(idx_eux, dtype=torch.long), torch.tensor(decal_eux[:-1], dtype=torch.long),
            torch.tensor(cibles, dtype=torch.float32))


# -- flux de données : dump officiel Lichess (FEN + évaluations Stockfish) -------------

URL_LICHESS_EVAL = "https://database.lichess.org/lichess_db_eval.jsonl.zst"


def lignes_lichess(url=URL_LICHESS_EVAL):
    """Générateur de lignes JSON décompressées à la volée (aucun fichier entier téléchargé d'un coup)."""
    import requests
    import zstandard
    with requests.get(url, stream=True, timeout=30) as reponse:
        reponse.raise_for_status()
        flux = zstandard.ZstdDecompressor().stream_reader(reponse.raw)
        for ligne in io.TextIOWrapper(flux, encoding="utf-8"):
            yield ligne


def parser_ligne(ligne, profondeur_min=20):
    """Une ligne du dump -> (fen, cp) ou None si filtrée (mauvaise qualité / format inattendu)."""
    try:
        d = json.loads(ligne)
        meilleur = max(d["evals"], key=lambda e: e.get("depth", 0))
        if meilleur.get("depth", 0) < profondeur_min:
            return None
        pv = meilleur["pvs"][0]
        if "mate" in pv:
            cp = 3000 if pv["mate"] > 0 else -3000
        else:
            cp = pv["cp"]
        return d["fen"], cp
    except Exception:
        return None


class FluxPositions(IterableDataset):
    """Position par position, encodées à la volée, sans tout charger en mémoire.

    Avec plusieurs workers (DataLoader num_workers > 0) : chaque worker ouvre son propre
    flux réseau (le téléchargement + décompression + parsing JSON était le vrai goulot
    d'étranglement, pas l'encodage) et ne garde qu'une position sur `num_workers`,
    de façon à couvrir le flux sans le dupliquer ni rien sauter.
    """

    def __init__(self, source_lignes, n_max, profondeur_min=20):
        self.source_lignes = source_lignes  # fonction sans argument -> générateur de lignes
        self.n_max = n_max
        self.profondeur_min = profondeur_min

    def __iter__(self):
        info = torch.utils.data.get_worker_info()
        id_worker, nb_workers = (0, 1) if info is None else (info.id, info.num_workers)
        n_max_worker = self.n_max // nb_workers + (1 if id_worker < self.n_max % nb_workers else 0)
        n_lu, n_garde = 0, 0
        for ligne in self.source_lignes():
            if n_garde >= n_max_worker:
                return
            if n_lu % nb_workers != id_worker:
                n_lu += 1
                continue
            n_lu += 1
            r = parser_ligne(ligne, self.profondeur_min)
            if r is None:
                continue
            fen, cp = r
            try:
                yield exemple(fen, cp)
            except Exception:
                continue  # FEN illisible ou position invalide : on ignore, on n'arrête pas tout
            n_garde += 1


# -- sauvegarde / reprise ---------------------------------------------------------------

def sauvegarder(chemin, modele, opt, n_vues, secondes):
    tmp = chemin + ".tmp"
    torch.save({"modele": modele.state_dict(), "opt": opt.state_dict(),
                "n_vues": n_vues, "secondes": secondes}, tmp)
    os.replace(tmp, chemin)


def charger(chemin, modele, opt=None, device="cpu"):
    pt = torch.load(chemin, map_location=device)
    modele.load_state_dict(pt["modele"])
    if opt is not None and "opt" in pt:
        opt.load_state_dict(pt["opt"])
    return pt.get("n_vues", 0), pt.get("secondes", 0.0)

## Réglages

In [ ]:
import sys, time
sys.path.insert(0, os.getcwd())
import torch
import commun_pown as pown

DOSSIER_DEPART = None       # ex : f"{SORTIE}/pown.pt" pour reprendre un entraînement précédent
NOM_SORTIE = "pown"
TIME_MIN = 280              # s'arrête proprement avant, sauvegarde faite régulièrement
N_POS = 100_000_000         # positions visées par passage (le temps peut couper avant)
EPOQUES = 3                 # nombre de passages sur le flux (si le temps le permet)
PROFONDEUR_MIN = 20         # comme PAWN : on ignore les évaluations Stockfish trop peu profondes
TAILLE_LOT = 16384
NB_WORKERS = 0                # testé : plusieurs workers retéléchargent chacun tout le fichier
                               # depuis le début (le vrai frein semble être la bande passante réseau,
                               # peut-être bridée côté Lichess) → plusieurs flux = plus lent, pas plus rapide
LR = 1e-3
SAUVEGARDE_SEC = 600        # point de contrôle toutes les 10 min
SEED = 42
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device :", DEVICE)
if DEVICE == "cpu":
    print("⚠️ pas de GPU : Runtime → Modifier le type d'exécution → GPU (T4)")

## Modèle (reprise si `DOSSIER_DEPART` est renseigné)

In [ ]:
modele = pown.ReseauPOWN().to(DEVICE)
opt = torch.optim.AdamW(modele.parameters(), lr=LR)
n_vues, secondes_avant = 0, 0.0
if DOSSIER_DEPART and os.path.exists(DOSSIER_DEPART):
    n_vues, secondes_avant = pown.charger(DOSSIER_DEPART, modele, opt, DEVICE)
    print(f"reprise : {n_vues:,} positions déjà vues, {secondes_avant/60:.0f} min déjà passées")
print("paramètres :", sum(t.numel() for t in modele.parameters()))

## Entraînement (flux Lichess, coupure au temps imparti)

In [ ]:
from torch.utils.data import DataLoader

CHEMIN = f"{SORTIE}/{NOM_SORTIE}.pt"
perte_fn = torch.nn.BCEWithLogitsLoss()
t0 = time.time()
derniere_sauvegarde = t0
pertes = []

for epoque in range(EPOQUES):
    if time.time() - t0 > TIME_MIN * 60:
        break
    print(f"--- passage {epoque + 1}/{EPOQUES} ---")
    flux = pown.FluxPositions(pown.lignes_lichess, n_max=N_POS, profondeur_min=PROFONDEUR_MIN)
    dl = DataLoader(flux, batch_size=TAILLE_LOT, collate_fn=pown.rassembler,
                     num_workers=NB_WORKERS, pin_memory=(DEVICE == "cuda"), prefetch_factor=(4 if NB_WORKERS else None))
    for i, (idx_n, dec_n, idx_e, dec_e, cibles) in enumerate(dl):
        non_bloquant = DEVICE == "cuda"
        idx_n, dec_n = idx_n.to(DEVICE, non_blocking=non_bloquant), dec_n.to(DEVICE, non_blocking=non_bloquant)
        idx_e, dec_e = idx_e.to(DEVICE, non_blocking=non_bloquant), dec_e.to(DEVICE, non_blocking=non_bloquant)
        cibles = cibles.to(DEVICE, non_blocking=non_bloquant)

        opt.zero_grad()
        sortie = modele(idx_n, dec_n, idx_e, dec_e)
        perte = perte_fn(sortie, cibles)
        perte.backward()
        opt.step()

        n_vues += cibles.shape[0]
        pertes.append(perte.item())

        if i % 200 == 0:
            vitesse = n_vues / (time.time() - t0 + 1e-9)
            print(f"{n_vues:>12,} positions | perte {sum(pertes[-200:]) / len(pertes[-200:]):.4f} "
                  f"| {vitesse:,.0f} pos/s | {(time.time() - t0) / 60:.0f} min")

        if time.time() - derniere_sauvegarde > SAUVEGARDE_SEC:
            pown.sauvegarder(CHEMIN, modele, opt, n_vues, secondes_avant + time.time() - t0)
            derniere_sauvegarde = time.time()

        if time.time() - t0 > TIME_MIN * 60:
            print("budget temps atteint, on s'arrête proprement")
            break

pown.sauvegarder(CHEMIN, modele, opt, n_vues, secondes_avant + time.time() - t0)
print(f"terminé : {n_vues:,} positions vues au total, modèle sauvegardé dans {CHEMIN}")

## Petit test de cohérence

In [ ]:
import chess

modele.eval()
print("position de départ :", round(modele.eval_cp(chess.STARTING_FEN), 1), "cp (attendu : proche de 0)")

# net avantage matériel pour les blancs (une Dame de plus)
print("blancs avec une Dame de plus :",
      round(modele.eval_cp("rnb1kbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1"), 1), "cp")
b_sans_dame_noire = chess.Board("rnb1kbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1")
print("(sans la dame noire, ce devrait être largement positif)")
modele.train()

### Et après ?
- **Pour continuer demain** : remets `DOSSIER_DEPART = f"{SORTIE}/pown.pt"` (le fichier est
  dans ton Drive, il survit à la session) et relance.
- **Pour le comparer aux autres modèles** : il faudra un petit pont ONNX ou un chargeur
  direct dans `commun_echecs.py`, plus une recherche alpha-bêta qui l'utilise — prochaine étape.
- Cette v1 mesure juste que **l'entraînement tourne et que la perte baisse**. Le vrai jugement
  viendra des benchmarks (comme pour PAWN, SPAWN, PWN) une fois qu'il pourra jouer.